# TimeSeries Forge — End-to-End Demo

This notebook walks through the full pipeline on synthetic multivariate sensor data:

1. Generate synthetic data with injected, labeled anomalies
2. Build chronological train/val/test splits and a sliding-window dataset
3. Train `ForgeNet` (a few epochs, small config, for a quick demo)
4. Evaluate forecast calibration and anomaly detection performance
5. Visualize variable-importance and attention weights for interpretability
6. Export to TorchScript for deployment

Run `pip install -e "..[dev,viz]"` from the repo root first if you haven't installed the package.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import torch
import matplotlib.pyplot as plt

from timeseries_forge.data.synthetic import generate_synthetic_series
from timeseries_forge.data.datasets import ChannelScaler, SlidingWindowDataset, train_val_test_split_indices
from timeseries_forge.models.forge_net import ForgeNet, ForgeNetConfig
from timeseries_forge.training.trainer import Trainer, TrainerConfig
from timeseries_forge.evaluation.forecast_metrics import evaluate_forecast
from timeseries_forge.evaluation.anomaly_metrics import evaluate_anomaly_detection
from timeseries_forge.utils.seed import set_seed
from torch.utils.data import DataLoader

set_seed(42)

## 1. Synthetic data with labeled anomalies

In [ ]:
data, anomaly_labels = generate_synthetic_series(n_steps=4000, n_channels=6, anomaly_rate=0.015)

fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(data[:500, i], lw=0.8)
    anomaly_idx = np.where(anomaly_labels[:500] == 1)[0]
    ax.scatter(anomaly_idx, data[anomaly_idx, i], color="red", s=10, zorder=5, label="anomaly")
    ax.set_ylabel(f"channel {i}")
axes[0].legend()
axes[-1].set_xlabel("timestep")
plt.suptitle("Synthetic multivariate series (first 500 steps) with injected anomalies")
plt.tight_layout()
plt.show()

## 2. Chronological split + scaling + windowing

In [ ]:
train_idx, val_idx, test_idx = train_val_test_split_indices(len(data))
train_raw, val_raw, test_raw = data[train_idx], data[val_idx], data[test_idx]
test_labels = anomaly_labels[test_idx]

scaler = ChannelScaler().fit(train_raw)  # fit on train ONLY -- no leakage
train_scaled = scaler.transform(train_raw)
val_scaled = scaler.transform(val_raw)
test_scaled = scaler.transform(test_raw)

SEQ_LEN, HORIZON, TARGETS = 96, 24, [0, 1]

train_ds = SlidingWindowDataset(train_scaled, SEQ_LEN, HORIZON, TARGETS, noise_std=0.05)
val_ds = SlidingWindowDataset(val_scaled, SEQ_LEN, HORIZON, TARGETS)
test_ds = SlidingWindowDataset(test_scaled, SEQ_LEN, HORIZON, TARGETS)

print(f"train/val/test windows: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}")

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

## 3. Build and train ForgeNet (small config + few epochs for a fast demo)

In [ ]:
config = ForgeNetConfig(
    num_features=data.shape[1],
    num_targets=len(TARGETS),
    seq_len=SEQ_LEN,
    horizon=HORIZON,
    d_model=64,
    n_heads=4,
    n_layers=2,
    ffn_hidden=128,
)
model = ForgeNet(config)
print(f"trainable parameters: {model.num_parameters():,}")

trainer_config = TrainerConfig(epochs=8, lr=3e-4, checkpoint_dir="demo_checkpoints", log_dir="demo_runs")
trainer = Trainer(model, trainer_config)
history = trainer.fit(train_loader, val_loader)

In [ ]:
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("total loss"); plt.legend(); plt.title("Training curves")
plt.show()

## 4. Evaluate forecast calibration + anomaly detection

In [ ]:
model.eval()
all_forecasts, all_targets, all_scores = [], [], []
with torch.no_grad():
    for batch in test_loader:
        out = model(batch["input"])
        all_forecasts.append(out["forecast"])
        all_targets.append(batch["forecast_target"])
        err = (out["reconstruction"] - batch["input"]).pow(2).mean(dim=-1)
        all_scores.append(err)

forecasts = torch.cat(all_forecasts, dim=0)
targets = torch.cat(all_targets, dim=0)
scores = torch.cat(all_scores, dim=0).numpy()

forecast_report = evaluate_forecast(targets, forecasts, config.quantiles)
print("Forecast metrics:")
for k, v in forecast_report.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
window_labels = np.array([test_labels[s + SEQ_LEN - 1] for s in test_ds.starts])
last_step_scores = scores[:, -1]

anomaly_report = evaluate_anomaly_detection(window_labels, last_step_scores)
print("Anomaly detection metrics:")
for k, v in anomaly_report.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## 5. Interpretability: variable selection + attention weights

One forward pass also returns the variable-selection weights (which
input channels mattered most at each timestep) and attention weights
(which past timesteps the model focused on) -- useful for explaining
predictions to non-ML stakeholders.

In [ ]:
sample_batch = next(iter(test_loader))
with torch.no_grad():
    out = model(sample_batch["input"][:1])

var_weights = out["var_weights"][0].numpy()  # (seq_len, num_features)
plt.figure(figsize=(10, 4))
plt.imshow(var_weights.T, aspect="auto", cmap="viridis")
plt.colorbar(label="selection weight")
plt.xlabel("timestep in window"); plt.ylabel("input channel")
plt.title("Variable Selection Network weights (one sample)")
plt.show()

last_layer_attn = out["attn_weights"][-1][0].mean(dim=0).numpy()  # average over heads
plt.figure(figsize=(6, 5))
plt.imshow(last_layer_attn, cmap="magma")
plt.colorbar(label="attention weight")
plt.xlabel("key timestep"); plt.ylabel("query timestep")
plt.title("Final-layer self-attention (averaged over heads)")
plt.show()

## 6. Export for deployment

In [ ]:
from timeseries_forge.deployment.export import export_torchscript

example_input = sample_batch["input"][:1]
path = export_torchscript(model, example_input, "../artifacts/model_demo.pt")
print(f"TorchScript artifact saved to {path}")